# 🎨 Pyxel Canvas
### 100 Stunning 2D & 3D Patterns in a Single Jupyter Notebook

---

Welcome to **Pyxel Canvas** — a fully interactive visual engine and gallery system.
Select a category, pick a pattern, adjust the controls, and click **RENDER**.

> **Tip:** Run all cells from top to bottom on first launch. Section 0 will install all dependencies automatically.

---
## Section 0 — Environment Setup
Install all required libraries and configure the notebook environment.

In [ ]:
# ── Section 0: Environment Setup ─────────────────────────────────
import subprocess, sys, importlib, os
from pathlib import Path

# Root path — all other paths are relative to this
NOTEBOOK_ROOT = Path.cwd()
print(f"📁 NOTEBOOK_ROOT = {NOTEBOOK_ROOT}")

# Required packages
_REQUIRED = [
    "numpy", "matplotlib", "Pillow", "ipywidgets",
    "scipy", "noise", "numba", "tqdm",
]

def _install(pkg):
    """Install a package if not already present."""
    try:
        importlib.import_module(pkg.split('[')[0].lower().replace('-', '_'))
        print(f"  ✅ {pkg} — already installed")
    except ImportError:
        print(f"  📦 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"  ✅ {pkg} — installed")

print("\n🔧 Checking dependencies...\n")
for pkg in _REQUIRED:
    _install(pkg)

# Create output directories
for d in ["exports", "assets", "shaders"]:
    (NOTEBOOK_ROOT / d).mkdir(exist_ok=True)

print("\n✨ Environment ready!")

---
## Section 1 — Core Engine
Load pattern registry, engines, and utilities into the notebook namespace.

In [ ]:
# ── Section 1: Core Engine ───────────────────────────────────────
import sys
from pathlib import Path

# Make sure our package is importable
_engine_root = str(Path.cwd().parent) if Path.cwd().name == 'visual_engine' else str(Path.cwd())
if _engine_root not in sys.path:
    sys.path.insert(0, _engine_root)

_ve_root = str(Path.cwd()) if Path.cwd().name == 'visual_engine' else str(Path.cwd() / 'visual_engine')
if _ve_root not in sys.path:
    sys.path.insert(0, _ve_root)

# Import the pattern registry
from patterns import PATTERNS, CATEGORIES
from engines import BasePattern, ColorUtils, PALETTES
from engines.color_utils import ColorUtils
from utils.export import export_png, export_gif, export_mp4

print(f"🎨 Loaded {len(PATTERNS)} patterns across {len(CATEGORIES)} categories")
for cat, pats in CATEGORIES.items():
    print(f"   • {cat}: {len(pats)} patterns")

---
## Section 2 — UI Controls
Interactive gallery interface — select a category and pattern, adjust controls, render.

In [ ]:
# ── Section 2: UI Controls ───────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# ── Lazy-loading cache ────────────────────────────────────────────
_render_cache = {}

def _get_renderer(name):
    """Lazy-load a pattern renderer."""
    if name not in _render_cache:
        _render_cache[name] = PATTERNS[name]
    return _render_cache[name]

# ── Widgets ────────────────────────────────────────────────────────

# Category dropdown
category_dd = widgets.Dropdown(
    options=list(CATEGORIES.keys()),
    value=list(CATEGORIES.keys())[0],
    description='Category:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='320px'),
)

# Pattern dropdown
pattern_dd = widgets.Dropdown(
    options=CATEGORIES[category_dd.value],
    description='Pattern:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='320px'),
)

# Universal controls
palette_dd = widgets.Dropdown(
    options=list(PALETTES.keys()),
    value='Inferno',
    description='Palette:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
)

speed_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description='Speed:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
    readout_format='.1f',
)

resolution_dd = widgets.Dropdown(
    options=['Low', 'Medium', 'High'],
    value='Low',
    description='Resolution:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
)

# Buttons
render_btn = widgets.Button(
    description='🎨 RENDER',
    button_style='success',
    layout=widgets.Layout(width='160px', height='40px'),
    style={'font_weight': 'bold'},
)

export_btn = widgets.Button(
    description='💾 EXPORT',
    button_style='info',
    layout=widgets.Layout(width='160px', height='40px'),
    style={'font_weight': 'bold'},
)

# Output areas
pattern_controls_area = widgets.Output(layout=widgets.Layout(
    border='1px solid #333', min_height='40px', padding='8px',
    margin='4px 0',
))
render_output = widgets.Output(layout=widgets.Layout(
    border='1px solid #444', min_height='200px', padding='8px',
))
status_label = widgets.HTML(
    value='<i style="color:#888;">Select a pattern and click RENDER</i>',
    layout=widgets.Layout(margin='4px 0'),
)

# ── Callbacks ──────────────────────────────────────────────────────

def _on_category_change(change):
    """Update pattern dropdown when category changes."""
    pattern_dd.options = CATEGORIES[change['new']]
    pattern_dd.value = CATEGORIES[change['new']][0]

def _on_pattern_change(change):
    """Update pattern-specific controls when pattern changes."""
    with pattern_controls_area:
        clear_output(wait=True)
        renderer = _get_renderer(change['new'])
        controls = renderer.get_controls()
        if controls:
            for ctrl in controls:
                display(ctrl)
        else:
            print('No pattern-specific controls.')

_last_fig = [None]  # mutable container for export

def _on_render(btn):
    """Render the selected pattern."""
    with render_output:
        clear_output(wait=True)
        name = pattern_dd.value
        status_label.value = f'<b style="color:#4fc3f7;">⏳ Rendering: {name}...</b>'
        renderer = _get_renderer(name)
        
        # Gather pattern-specific control values
        extra_kwargs = {}
        controls = renderer.get_controls()
        for ctrl in controls:
            if hasattr(ctrl, 'description') and hasattr(ctrl, 'value'):
                key = ctrl.description.strip(':').strip().lower().replace(' ', '_')
                extra_kwargs[key] = ctrl.value
        
        try:
            renderer.render(
                resolution=resolution_dd.value,
                palette=palette_dd.value,
                speed=speed_slider.value,
                **extra_kwargs,
            )
            # Try to capture current figure for export
            figs = [plt.figure(i) for i in plt.get_fignums()]
            _last_fig[0] = figs[-1] if figs else None
            status_label.value = f'<b style="color:#81c784;">✅ Rendered: {name}</b>'
        except Exception as e:
            status_label.value = f'<b style="color:#e57373;">❌ Error: {e}</b>'
            import traceback; traceback.print_exc()

def _on_export(btn):
    """Export the last rendered figure."""
    if _last_fig[0] is not None:
        path = export_png(_last_fig[0], pattern_dd.value)
        status_label.value = f'<b style="color:#81c784;">💾 Exported → {path}</b>'
    else:
        status_label.value = '<b style="color:#ffb74d;">⚠️ Nothing to export — render a pattern first.</b>'

# Wire up callbacks
category_dd.observe(_on_category_change, names='value')
pattern_dd.observe(_on_pattern_change, names='value')
render_btn.on_click(_on_render)
export_btn.on_click(_on_export)

# ── Layout ─────────────────────────────────────────────────────────

header = widgets.HTML(
    value='''
    <div style="background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
                padding:16px 24px; border-radius:12px; margin-bottom:12px;">
        <h2 style="color:#e0e0e0; margin:0;">🎨 Pyxel Canvas</h2>
        <p style="color:#888; margin:4px 0 0;">Select a category → pick a pattern → click RENDER</p>
    </div>
    '''
)

selectors_row = widgets.HBox([category_dd, pattern_dd],
                              layout=widgets.Layout(gap='12px'))
controls_row = widgets.HBox([palette_dd, speed_slider, resolution_dd],
                             layout=widgets.Layout(gap='12px'))
buttons_row = widgets.HBox([render_btn, export_btn],
                            layout=widgets.Layout(gap='12px', justify_content='flex-start'))

ui = widgets.VBox([
    header,
    selectors_row,
    controls_row,
    pattern_controls_area,
    buttons_row,
    status_label,
    render_output,
], layout=widgets.Layout(padding='8px'))

display(ui)

# Trigger initial pattern controls display
_on_pattern_change({'new': pattern_dd.value})

---
## Section 3 — Render Section
The RENDER button above triggers rendering. No separate cell needed — the UI handles it.

---

## Section 4 — Pattern Implementations

All 100 patterns are implemented as classes in the `patterns/` module files.
Currently all patterns are stubs showing placeholder cards.
As patterns are implemented during Phase 1, their Concept Note cells will be added below.

---

## Section 5 — Export Utilities

Export functions are available via the **💾 EXPORT** button above.
- `export_png(figure, name)` — saves as PNG
- `export_gif(frames, name, fps)` — saves as animated GIF
- `export_mp4(frames, name, fps)` — saves as MP4 (falls back to GIF if ffmpeg unavailable)

All outputs are saved to `visual_engine/exports/`.